# Data Description
- Background: 第七輪核對9人的當屆履歷與2022公報身分，另查9個政黨標示人物頁。
- date received: 20261019
- Path to data file: 下方固定PINS逐檔列示；原始網址與擷取方式在3份manifest。
- Unit of observation: HTTP嘗試、工具擷取頁、原始公報PDF各自保留，不混算成人數。
- Sample period: 2022公報及2026報導／競選頁；2026-10-19擷取。
- Known issues: HTTP403／404不是可用內文；kingtop雖200但只有BOM，不作證據。web工具文字不是原始HTML，也不宣稱HTTP200；兩個工具開啟失敗保留為失敗紀錄。9個政黨標示頁僅3頁有實質背景，6頁占位；網站政黨標示不是政府學位認證。
- Definition for each variable: manifest的candidate_name為查找對象；url/requested_reference為來源；status為實際HTTP結果；path/sha256為保存檔與位元組身分；retrieval_method區分工具文字與HTTP回應；has_substantive_background表示頁面實際有學經歷。
- Data path: PINS全部固定路徑。
- Role: raw source bytes / retrieval metadata / web-tool text，逐類保留。
- Verification status: 執行下方逐檔存在與SHA檢查；不修改來源。
- SHA-256: PINS逐檔列示。
- Next processing notebook: [第七輪處理](../code/council_identity_round7_20261019.ipynb)。人工審閱設定另在data/code/identity_followup_review_20261019.json，不冒充原始資料。


In [1]:
from pathlib import Path
import json, hashlib
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'data').exists())
sha=lambda p:hashlib.sha256((ROOT/p).read_bytes()).hexdigest()
PINS={'data/raw/background_expansion_20261019/0108098a8b75f3cadb7a.source': '737d6a547b3012138fccb0e2d1375aea3415d7128d5842ef3f795f28f0bc5755', 'data/raw/background_expansion_20261019/06a729e8f2a6e8dd5b9e.source': '13d98d3897822d0ca5de04681a42337d7d3dc84443f965ae1d8f388f4692ecf0', 'data/raw/background_expansion_20261019/07d3192d368be5a7d26c.source': '22ae5353195c3fe1335d3a93ec34a94a52a8ad38b59c7c1ebc6cffc59cce07fe', 'data/raw/background_expansion_20261019/5a9df4305aba6e00ab63.source': '6b80caadfefe6397e93270e076ea99f7d4cbd175a54fc6aecbb726e215c53d9d', 'data/raw/background_expansion_20261019/806bb898ba5b648d29f0.source': 'fc917d512e7aa3c652fb4dae815311f054fc6ea91ac31583d8e72e68f9d616bc', 'data/raw/council_review_20261019/xu_bulletin_2022.pdf': 'f235d01f3f55f226a269fe4fa002bd2ad068b00d2d34e6c7a97d963e6fa552e5', 'data/raw/background_expansion_20261019/a836ab3e1f22f8591f85.source': '9c8499711889138b7a5a83daa96c78d98a19a97db16cee8709f9d84dd1b60018', 'data/raw/background_expansion_20261019/b141e9c68065df0bc099.source': 'd6386bdde3161f2247a2e6337aac62b40926e609f7476801b3dbf95dfe3a90b8', 'data/raw/background_expansion_20261019/bd66678932406109abe6.source': '9427a9b23315effeada0c78fbfc63d55d4823e2d9b462dca0e7e96ccd6daf54d', 'data/raw/identity_followup_20261019/005f873a8fc98fe29510.html': '116c06ad0f6a87a000d965dd969271978b57d6e008c6495810ea01b2ac55e537', 'data/raw/mayor_birth_followup_20261019/wen_cna.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_followup_20261019/1bbf09cf78013308abf9.html': '9882c37014940b4b8c53ba25eb58c8407423d4c8340b1e63ad1707f5ef5ac972', 'data/raw/identity_followup_20261019/2284ca924ca216df5ac4.html': '59d65954f2f1596d2ea89957423b48c48af18eb3d4d8ca9ae891322611fb2ec9', 'data/raw/identity_followup_20261019/2e04933edda8d0e24c09.html': 'ea838d8d45e7a770c5544113bc98bd4aa2475d750c22329443de4be5b05ce939', 'data/raw/mayor_birth_followup_20261019/wen_cna.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_followup_20261019/64fbcee5c646c495e8a3.html': '17ea8d8c4f070d0b6fd4c08935fcaae7553667f43acfdd8cb587a090d867e581', 'data/raw/identity_followup_20261019/64fbcee5c646c495e8a3.html': '17ea8d8c4f070d0b6fd4c08935fcaae7553667f43acfdd8cb587a090d867e581', 'data/raw/identity_followup_20261019/b3a33c911ca5f0207e34.html': '7271c9298115d75e7648fd3fe3721d21b9c4c16d13ced7bbfd0187a37c3469d5', 'data/raw/identity_followup_20261019/bca9b294ee6dbb84749f.html': 'ce3e26762cf67d679c0b0e160fb3f36fd4e027e2e9228fa37bfebea830dfd381', 'data/raw/identity_followup_20261019/c1e0df6c4a9d33cee592.html': '764280c179712fff20dd8c57b7c6cbb6c1694313942710ae1306fe5b9fc8345e', 'data/raw/identity_followup_20261019/c74545cc882cae9c26c0.html': '84599bd6035bf395187cae3f790bd8fd2a9f00ce0250f4e5df3f4e76457d95f2', 'data/raw/identity_followup_20261019/e6161ed772261edbe0f8.html': '5d4caceaa45a7d3c214bd99a138d4c6e72843962ab9221d8a0330316ecc6a2bf', 'data/raw/identity_followup_20261019/manifest.json': '1c1a227e2612c53e56892604fcb4f6e5c2d63bb7c77957551c02911cb61d1d34', 'data/raw/identity_followup_20261019/web_01.txt': '5311e75d0896fbc18e58a6d9465e94fab7d6c99cfe7e0460fc6c81bda040576b', 'data/raw/identity_followup_20261019/web_02.txt': '999edfa5ba3be1c901aaf8ca935218c11c0e302105e19b5f0e196f328549637a', 'data/raw/identity_followup_20261019/web_03.txt': '8c3ca40774032c94b68e800f052a2246e7d040c82915a998373a4876e01dd2ff', 'data/raw/identity_followup_20261019/web_04.txt': '914d34c2ac8722210d5d39be59648c1bb414d752632d097184b961eaef816a8d', 'data/raw/identity_followup_20261019/web_05.txt': 'dbbfca46230c29e4ef8b8f5c813ef017a7d2da82ca668f850da706f639c4709e', 'data/raw/identity_followup_20261019/web_manifest.json': '834b0328d6458eaf7c1138e5466db42771c57e5a0b9219590581f7d300c7ea3f', 'data/raw/kmtks_web_20261019/01.txt': '53604782ff866b558f1abfd3113c85eb442d0527900666d4b8851c5051bdc630', 'data/raw/kmtks_web_20261019/02.txt': 'a4c781c32e4519c4ef10b588df820d7905bb1a46c9d09d51f81fb61932f5a44f', 'data/raw/kmtks_web_20261019/03.txt': '112f40fd60c335821c9f82e85d1bf523b01653562ead8cad557ab65622767d3b', 'data/raw/kmtks_web_20261019/04.txt': 'bc90f2b6672fdfca62b6525175d89f6587b52755bc79edd3b703d8060f293fd7', 'data/raw/kmtks_web_20261019/05.txt': '410ff727927bf36c12bc505782951ead7cfb91b88b0e1d28e4802249c7f99b26', 'data/raw/kmtks_web_20261019/06.txt': 'ac238fc1e68800d9982db5f3490611b9db2504239e8bdf03a6561ae1269f4387', 'data/raw/kmtks_web_20261019/07.txt': 'a6968a6f31555c4549d683760deb30c380338543fca2e273c94ba077a89b2dbc', 'data/raw/kmtks_web_20261019/08.txt': '3dc649a66518b36895e1a2a3059808bd39a410b9d99f6d1baec7f07d0ba2c904', 'data/raw/kmtks_web_20261019/09.txt': '148790086000d925b5ec7d43a05a6503ec4fad6f6a63c5ff92b96f45aa9cb453', 'data/raw/kmtks_web_20261019/manifest.json': 'a519c5d80a86591bec6da77b7f031e56c5c10fa9e79245e5e1b74332a4709fb8'}
for p,h in PINS.items(): assert sha(p)==h,p
print("Source files checked:",len(PINS))

Source files checked: 39


## Change for Your Own Data
固定來源與hash；不下載、不補替代檔。

## Summary

### Sample

In [2]:
MFILES=['data/raw/identity_followup_20261019/manifest.json','data/raw/identity_followup_20261019/web_manifest.json','data/raw/kmtks_web_20261019/manifest.json']
for p in MFILES:
 d=pd.DataFrame(json.loads((ROOT/p).read_text()));print(p);display(d.head(10))

data/raw/identity_followup_20261019/manifest.json


,candidate_name,url,status,content_type,final_url,path,sha256,checked_date
0,林慧欣,https://newtalk.tw/news/view/2026-02-10/1019657,403,text/html; charset=UTF-8,https://newtalk.tw/news/view/2026-02-10/1019657,data/raw/identity_followup_20261019/1bbf09cf78...,9882c37014940b4b8c53ba25eb58c8407423d4c8340b1e...,2026-10-19
1,鄭東元,https://hoomedia.com.tw/2026/09/04/165433/,403,text/html; charset=UTF-8,https://hoomedia.com.tw/2026/09/04/165433/,data/raw/identity_followup_20261019/e6161ed772...,5d4caceaa45a7d3c214bd99a138d4c6e72843962ab9221...,2026-10-19
2,李柏融,https://tw.news.yahoo.com/高雄二選區-2無黨攪亂藍綠版圖-2010...,404,text/html,https://tw.news.yahoo.com/%E9%AB%98%E9%9B%84%E...,data/raw/identity_followup_20261019/64fbcee5c6...,17ea8d8c4f070d0b6fd4c08935fcaae7553667f43acfdd...,2026-10-19
3,柳淑芳,https://tw.news.yahoo.com/高雄十大傳說-舉牌姊-將八度參選-柳淑芳...,404,text/html,https://tw.news.yahoo.com/%E9%AB%98%E9%9B%84%E...,data/raw/identity_followup_20261019/a898038ddb...,17ea8d8c4f070d0b6fd4c08935fcaae7553667f43acfdd...,2026-10-19
4,陳惠君,https://more-news.tw/714185/,200,text/html; charset=UTF-8,https://more-news.tw/714185/,data/raw/identity_followup_20261019/c1e0df6c4a...,764280c179712fff20dd8c57b7c6cbb6c1694313942710...,2026-10-19
5,童燕珍,https://www.cna.com.tw/news/aloc/202604220132....,403,text/html; charset=UTF-8,https://www.cna.com.tw/news/aloc/202604220132....,data/raw/identity_followup_20261019/4e383d63bf...,c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42f...,2026-10-19
6,童燕珍,https://news.pchome.com.tw/living/cna/20260422...,403,text/html; charset=UTF-8,https://news.pchome.com.tw/living/cna/20260422...,data/raw/identity_followup_20261019/0d72576b6c...,c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42f...,2026-10-19
7,蘇致榮,https://newtalk.tw/news/view/2026-02-10/1019604,403,text/html; charset=UTF-8,https://newtalk.tw/news/view/2026-02-10/1019604,data/raw/identity_followup_20261019/bca9b294ee...,ce3e26762cf67d679c0b0e160fb3f36fd4e027e2e9228f...,2026-10-19
8,蘇致榮、徐尚賢,https://udn.com/vote2026/story/124652/9652381,200,text/html; charset=utf-8,https://udn.com/vote2026/story/124652/9652381,data/raw/identity_followup_20261019/005f873a8f...,116c06ad0f6a87a000d965dd969271978b57d6e008c649...,2026-10-19
9,徐尚賢,https://www.kingtop.com.tw/detail.php?id=31299...,200,text/html; charset=utf-8,https://www.kingtop.com.tw/detail.php?id=31299...,data/raw/identity_followup_20261019/c74545cc88...,84599bd6035bf395187cae3f790bd8fd2a9f00ce0250f4...,2026-10-19


data/raw/identity_followup_20261019/web_manifest.json


,candidate_name,requested_reference,retrieval_method,url,path,sha256,checked_date
0,林慧欣,https://newtalk.tw/news/view/2026-02-10/1019657,web_tool_extraction_not_original_HTTP,https://newtalk.tw/news/view/2026-02-10/1019657,data/raw/identity_followup_20261019/web_01.txt,5311e75d0896fbc18e58a6d9465e94fab7d6c99cfe7e04...,2026-10-19
1,鄭東元,https://hoomedia.com.tw/2026/09/04/165433/,web_tool_extraction_not_original_HTTP,https://hoomedia.com.tw/2026/09/04/165433/,data/raw/identity_followup_20261019/web_02.txt,999edfa5ba3be1c901aaf8ca935218c11c0e302105e19b...,2026-10-19
2,童燕珍,https://www.cna.com.tw/news/aloc/202604220132....,web_tool_extraction_not_original_HTTP,https://www.cna.com.tw/news/aloc/202604220132....,data/raw/identity_followup_20261019/web_03.txt,8c3ca40774032c94b68e800f052a2246e7d040c82915a9...,2026-10-19
3,李柏融,turn111search8,web_tool_extraction_not_original_HTTP,,data/raw/identity_followup_20261019/web_04.txt,914d34c2ac8722210d5d39be59648c1bb414d752632d09...,2026-10-19
4,柳淑芳,turn110search2,web_tool_extraction_not_original_HTTP,,data/raw/identity_followup_20261019/web_05.txt,dbbfca46230c29e4ef8b8f5c813ef017a7d2da82ca668f...,2026-10-19


data/raw/kmtks_web_20261019/manifest.json


,candidate_name,index_link_id,retrieval_method,path,sha256,checked_date,url,contains_placeholder,has_substantive_background
0,李振豪,59,web_tool_extraction_not_original_HTTP,data/raw/kmtks_web_20261019/01.txt,53604782ff866b558f1abfd3113c85eb442d0527900666...,2026-10-19,https://2026.kmtks.org.tw/Council/li-zhen-hao,True,False
1,黃韻涵,67,web_tool_extraction_not_original_HTTP,data/raw/kmtks_web_20261019/02.txt,a4c781c32e4519c4ef10b588df820d7905bb1a46c9d09d...,2026-10-19,https://2026.kmtks.org.tw/Council/huang-yun-han,True,False
2,陳若翠,81,web_tool_extraction_not_original_HTTP,data/raw/kmtks_web_20261019/03.txt,112f40fd60c335821c9f82e85d1bf523b01653562ead8c...,2026-10-19,https://2026.kmtks.org.tw/Council/chen-ruo-cui,True,False
3,童燕珍,91,web_tool_extraction_not_original_HTTP,data/raw/kmtks_web_20261019/04.txt,bc90f2b6672fdfca62b6525175d89f6587b52755bc79ed...,2026-10-19,https://2026.kmtks.org.tw/Council/tong-yan-zhen,True,False
4,黃裕庭,95,web_tool_extraction_not_original_HTTP,data/raw/kmtks_web_20261019/05.txt,410ff727927bf36c12bc505782951ead7cfb91b88b0e1d...,2026-10-19,https://2026.kmtks.org.tw/Council/huang-yu-ting,False,True
5,湯茹婷,109,web_tool_extraction_not_original_HTTP,data/raw/kmtks_web_20261019/06.txt,ac238fc1e68800d9982db5f3490611b9db2504239e8bdf...,2026-10-19,https://2026.kmtks.org.tw/Council/tang-ju-ting,False,True
6,宋紹銘,119,web_tool_extraction_not_original_HTTP,data/raw/kmtks_web_20261019/07.txt,a6968a6f31555c4549d683760deb30c380338543fca2e2...,2026-10-19,https://2026.kmtks.org.tw/Council/song-shao-ming,True,True
7,葛姵瑩,121,web_tool_extraction_not_original_HTTP,data/raw/kmtks_web_20261019/08.txt,3dc649a66518b36895e1a2a3059808bd39a410b9d99f6d...,2026-10-19,https://2026.kmtks.org.tw/Council/ge-pei-ying,True,False
8,陳奕蓁,125,web_tool_extraction_not_original_HTTP,data/raw/kmtks_web_20261019/09.txt,148790086000d925b5ec7d43a05a6503ec4fad6f6a63c5...,2026-10-19,https://2026.kmtks.org.tw/Council/chen-yi-zhen,True,False


### Summary Stats

In [3]:
http=pd.DataFrame(json.loads((ROOT/MFILES[0]).read_text()));party=pd.DataFrame(json.loads((ROOT/MFILES[2]).read_text()))
display(http.status.value_counts(dropna=False));display(party[['candidate_name','has_substantive_background','contains_placeholder']]);print('HTTP duplicate URLs:',http.url.duplicated().sum());display(http.isna().sum().to_frame('missing'));display(http.dtypes.to_frame('type'))
assert len(http)==13 and len(party)==9 and party.has_substantive_background.sum()==3

status
200    6
403    5
404    2
Name: count, dtype: int64

,candidate_name,has_substantive_background,contains_placeholder
0,李振豪,False,True
1,黃韻涵,False,True
2,陳若翠,False,True
3,童燕珍,False,True
4,黃裕庭,True,False
5,湯茹婷,True,False
6,宋紹銘,True,True
7,葛姵瑩,False,True
8,陳奕蓁,False,True


HTTP duplicate URLs: 0


,missing
candidate_name,0
url,0
status,0
content_type,0
final_url,0
path,0
sha256,0
checked_date,0


,type
candidate_name,object
url,object
status,object
content_type,object
final_url,object
path,object
sha256,object
checked_date,object


2026-10-20 維護：重複檔案改用單一保存路徑，固定指紋同步更新；議員登記CSV排版換行改為空格，政黨標籤不變。本次已獨立從raw表格重建並比對全部1,502筆及16欄；未重跑本Notebook或整條研究流程，既有執行輸出屬先前執行紀錄。